# Kaggle fine-tune: Solidity hunter (QLoRA on `splits/train_source.jsonl`)

One-click training notebook for the [smartcontractshacking](https://github.com/lipon101/smartcontractshacking) corpus.
It consumes **`splits/train_source.jsonl` directly** — including the GitHub 100 MB byte-split `.partNN` layout —
and formats every record into the **strict-JSON answer format** already defined by the repo's
[`scripts/eval_harness.py`](https://github.com/lipon101/smartcontractshacking/blob/main/scripts/eval_harness.py)
(SYSTEM_PROMPT + `extract_json`), so the fine-tuned model speaks exactly the dialect the Echidna gate expects.

**Answer contract (verbatim from `eval_harness.py`):**

```json
{"vulnerable": true/false, "vuln_type": "...", "severity": "critical|high|medium|low|gas",
 "poc": "<step-by-step exploit sequence>", "fix": "<what the fix is and why>",
 "patched_function": "<the FULL replacement function including its signature, or null if not vulnerable>"}
```

**How to run on Kaggle (one click):**
1. Create a Kaggle Dataset from the repo's `splits/` folder (upload the `train_source.jsonl.part00..03` + `eval_source.jsonl` files as-is; the loader reassembles them). Name it anything — the notebook auto-discovers `/kaggle/input/*/splits`.
2. New Notebook → **Add Input** → your dataset → Runtime: **GPU T4 x2** (16 GB).
3. **Run All.** Training (~1–2 h for Qwen3.5-9B QLoRA, 2 epochs) then an eval cell reports strict-JSON accuracy on the held-out `eval_source.jsonl` split.

**Model** (verified on HF, 2026-08-03; see `docs/qwen36-exact-links-and-tools.md`):
- Default `Qwen/Qwen3.5-9B` — fits the Kaggle T4 16 GB (docs: 16 GB fallback).
- `Qwen/Qwen3.6-27B` (Apache-2.0, 24 GB min) or the repo's stated target `samscrack/Qwen3.6-Solidity-27B` / `crichalchemist/Qwen3.6-Solidity-27B` — set `MODEL_ID` below and use a ≥24 GB GPU.

**After training:** `vps_setup.sh` serves the GGUF via Ollama; `eval_harness.py --model hunter --contract x.sol --function f` runs the differential-fuzz gate.


In [ ]:
# 1) Environment — Kaggle images already ship torch/transformers; install the rest.
#    NOTE: never install CUDA wheels on Kaggle; the prebuilt image has them.
import subprocess, sys

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs), check=True)

_pip("transformers", "accelerate", "peft", "trl", "bitsandbytes", "datasets")

import torch, transformers, accelerate
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| accelerate", accelerate.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU)")


In [ ]:
# 2) Configuration — model, paths, data policy, training hyperparameters.

import os, glob, json, hashlib, random
from pathlib import Path
from collections import Counter

SEED = 42
random.seed(SEED)

# ---------------- corpus paths ----------------
# Auto-discovery order: $CORPUS_DIR -> /kaggle/input/<any-dataset>/splits
# -> /kaggle/input/<any-dataset>/ (files loose in dataset root) -> local clone.
DATA_DIR = os.environ.get("CORPUS_DIR", "")
if not DATA_DIR:
    candidates = sorted(glob.glob("/kaggle/input/*/splits")) + sorted(glob.glob("/kaggle/input/*/"))
    candidates += [str(Path.cwd() / "splits"), str(Path.cwd().parent / "splits")]
    for c in candidates:
        if os.path.isdir(c) and glob.glob(os.path.join(c, "train_source.jsonl*")):
            DATA_DIR = c
            break
assert DATA_DIR, "Could not find the corpus. Set CORPUS_DIR=<path> or mount the Kaggle dataset."
print("DATA_DIR =", DATA_DIR)

TRAIN_GLOB = "train_source.jsonl*"   # 4 byte-split parts OR one reassembled file — both work
EVAL_FILE  = "eval_source.jsonl"

# SHA256 of the reassembled originals (README.md) — the loader verifies byte-exact integrity.
EXPECTED_SHA = {
    "train_source.jsonl": "a8fea891370879fa143ba15e4a12ff586fa51069318dd4d9ad31ad03e999012e",
    "eval_source.jsonl":  "312c14932754a9e56c0755d9196b1b4289fd8ca740eb147c18bad19deb8e6b40",
}
VERIFY_SHA = True
MAX_ROWS   = None          # optional row cap for local testing only

# ---------------- model / fine-tune ----------------
# T4 (16 GB) one-click default: Qwen3.5-9B. 27B variants need >=24 GB VRAM:
#   "Qwen/Qwen3.6-27B"                      base model (Apache-2.0, the docs pick)
#   "samscrack/Qwen3.6-Solidity-27B"        repo's stated Kaggle target (kaggle_ready_report.json)
#   "crichalchemist/Qwen3.6-Solidity-27B"   leaderboard-topping Solidity checkpoint
MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen3.5-9B")

MAX_SEQ_LEN       = 4096   # context cap; source/audit text is truncated to fit
MAX_CONTEXT_CHARS = 6000   # chars of contract source / audit narrative shown to the model
MAX_NEW_TOKENS    = 512    # answer length cap during eval generation

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]

EPOCHS           = 2
LR               = 2e-4
PER_DEVICE_BATCH = 1
GRAD_ACCUM       = 16      # effective batch = 1 * 16 = 16
WARMUP_RATIO     = 0.03
EVAL_MAX_ROWS    = 400     # rows of eval_source used for validation metrics
EVAL_N_ROWS      = 50      # rows used for the post-training strict-JSON eval
OUTPUT_DIR       = "/kaggle/working/hunter-qlora"

print("MODEL_ID =", MODEL_ID, "| MAX_SEQ_LEN =", MAX_SEQ_LEN)


In [ ]:
# 3) Loader — consume splits/train_source.jsonl DIRECTLY (byte-split parts included).
#    The .partNN files are byte-splits, NOT line-splits: 3 records straddle part
#    boundaries. We therefore stream the concatenated byte stream (cat part* in
#    sorted order), which is the only correct way to parse them, and verify the
#    published SHA256.

def cat_lines(files):
    '''Yield decoded lines of the byte-concatenated stream (true `cat part*` semantics:
    a record straddling a part boundary is reassembled before line-splitting).'''
    buf = b""
    for f in files:
        with open(f, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                buf += chunk
                while b"\n" in buf:
                    line, buf = buf.split(b"\n", 1)
                    yield line.decode("utf-8")
    if buf:
        yield buf.decode("utf-8")

def load_jsonl_parts(glob_pattern, verify_sha=None):
    '''Reassemble byte-split parts (cat part* in sorted order) and parse one JSON per line.'''
    files = sorted(glob.glob(os.path.join(DATA_DIR, glob_pattern)))
    assert files, f"no files match {DATA_DIR}/{glob_pattern}"
    print(f"reading {len(files)} file(s): {[os.path.basename(f) for f in files]}")

    if verify_sha:
        h = hashlib.sha256()
        for f in files:
            with open(f, "rb") as fh:
                for chunk in iter(lambda: fh.read(1 << 20), b""):
                    h.update(chunk)
        got = h.hexdigest()
        print(f"sha256(concat) = {got}")
        assert got == verify_sha, f"SHA256 MISMATCH — expected {verify_sha}, got {got}"

    rows = []
    for line in cat_lines(files):
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
        if MAX_ROWS and len(rows) >= MAX_ROWS:
            return rows
    return rows

train_rows = load_jsonl_parts(TRAIN_GLOB, EXPECTED_SHA["train_source.jsonl"] if VERIFY_SHA else None)
eval_rows  = load_jsonl_parts(EVAL_FILE,  EXPECTED_SHA["eval_source.jsonl"]  if VERIFY_SHA else None)

print(f"train = {len(train_rows)} records | eval = {len(eval_rows)} records")
print("train keys:", sorted(train_rows[0].keys()))


In [ ]:
# 4) Leakage & stats — the repo ships a protocol-exclusive split; never re-split randomly.
#    (Re-splitting train_source.jsonl at random would leak protocols into validation.)

train_ids, eval_ids   = {r["id"] for r in train_rows}, {r["id"] for r in eval_rows}
train_prot, eval_prot = {r["protocol"] for r in train_rows}, {r["protocol"] for r in eval_rows}
assert train_ids.isdisjoint(eval_ids),   "FAIL: id overlap between train and eval!"
assert train_prot.isdisjoint(eval_prot), "FAIL: protocol overlap between train and eval!"
print(f"id overlap       = {len(train_ids & eval_ids)}")
print(f"protocol overlap = {len(train_prot & eval_prot)}  (train {len(train_prot)} / eval {len(eval_prot)})")

print("train severity:", dict(Counter(r["severity"] for r in train_rows)))
print("eval  severity:", dict(Counter(r["severity"] for r in eval_rows)))
print(f"train with source   : {sum(1 for r in train_rows if r['source'])}/{len(train_rows)}")
print(f"train with function : {sum(1 for r in train_rows if r['function'])}/{len(train_rows)}")
print(f"train empty vuln_type: {sum(1 for r in train_rows if not r['vuln_type'].strip())}")


In [ ]:
# 5) Format — every record becomes a chat triple ending in the STRICT-JSON answer.
#    SYSTEM_PROMPT mirrors scripts/eval_harness.py verbatim (the repo's single
#    source of truth for the answer contract). Severity is normalized from the
#    corpus enum (LOW/MEDIUM/HIGH/GAS) to the harness enum (low/medium/high/gas).
#    Answers are serialized with json.dumps(ensure_ascii=False) — never hand-built.

SYSTEM_PROMPT = (
    "You are a professional smart-contract security auditor for bug bounties (Immunefi). "
    "Given a Solidity contract and a function, find the vulnerability, prove it with a "
    "concrete exploit PoC, and give the exact fix. Output ONLY strict JSON:\n"
    '{"vulnerable": true/false, "vuln_type": "...", "severity": "critical|high|medium|low|gas", '
    '"poc": "<step-by-step exploit sequence>", "fix": "<what the fix is and why>", '
    '"patched_function": "<the FULL replacement function including its signature, '
    'or null if not vulnerable>"}'
)

SEVERITY_MAP = {"LOW": "low", "MEDIUM": "medium", "HIGH": "high", "GAS": "gas"}

def normalize_severity(s):
    return SEVERITY_MAP.get(s, (s or "low").lower())

def build_context(r, max_chars=MAX_CONTEXT_CHARS):
    '''Contract context: full source when available, else the vulnerable-function
    excerpt, else the audit narrative — truncated to fit the context window.'''
    if r.get("source"):
        ctx = r["source"]
    elif r.get("function"):
        ctx = r["function"]
    else:
        ctx = r.get("audit_text", "")
    return ctx[:max_chars] + ("..." if len(ctx) > max_chars else "")

def build_messages(r):
    user = (f"Contract `{r.get('contract_name') or 'unknown'}`:\n\n"
            f"{build_context(r)}\n\n"
            f"Audit this contract. Finding id: {r['id']}.")
    answer = {
        "vulnerable": bool(r.get("is_real", True)),
        "vuln_type": r.get("vuln_type", ""),
        "severity": normalize_severity(r.get("severity", "")),
        "poc": r.get("poc", ""),
        "fix": r.get("fix", ""),
        "patched_function": None,  # corpus fixes are prose, not full replacement functions
    }
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": json.dumps(answer, ensure_ascii=False)},
    ]

def extract_json(text):
    '''Same parser as scripts/eval_harness.py: first {...} block, json.loads.'''
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("no JSON object in model output")
    return json.loads(text[start:end + 1])

def score_answer(gold, pred):
    '''gold/pred: strict-JSON dicts. Returns per-key correctness.'''
    gv, pv = (gold.get("vuln_type") or "").strip().lower(), (pred.get("vuln_type") or "").strip().lower()
    return {
        "json_ok": True,
        "vulnerable": gold.get("vulnerable") is pred.get("vulnerable"),
        "severity": gold.get("severity") == pred.get("severity"),
        "vuln_type": bool(gv) and (gv == pv or gv in pv or pv in gv),
    }

# ---- build training/eval record lists (drop the 56 empty-vuln_type records) ----
train_records = [r for r in train_rows if r["vuln_type"].strip()]
eval_records  = [r for r in eval_rows if r["vuln_type"].strip()]
print(f"train records after drop-empty-vuln_type: {len(train_records)} (dropped {len(train_rows) - len(train_records)})")
print(f"eval  records after drop-empty-vuln_type: {len(eval_records)} (dropped {len(eval_rows) - len(eval_records)})")

# sanity: every answer is strict JSON and round-trips through the harness parser
for r in train_records[:3] + [train_records[len(train_records) // 2]]:
    msgs = build_messages(r)
    gold = json.loads(msgs[-1]["content"])
    assert extract_json(msgs[-1]["content"]) == gold, "answer does not round-trip through extract_json"
    print("example answer:", json.dumps(gold, ensure_ascii=False)[:200])


In [ ]:
# 6) Tokenizer + datasets — chat-template the records; SFTTrainer tokenizes lazily.

from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_for_sft(record):
    return tokenizer.apply_chat_template(build_messages(record), tokenize=False, add_generation_prompt=False)

train_ds = Dataset.from_list(train_records)
eval_ds  = Dataset.from_list(eval_records[:EVAL_MAX_ROWS])
print(f"train_ds = {len(train_ds)} rows | eval_ds = {len(eval_ds)} rows")

sample = format_for_sft(train_records[0])
print("formatted example chars:", len(sample))
print(sample[:700].replace("\n", "\n"))


In [ ]:
# 7) QLoRA setup — 4-bit NF4 base + LoRA adapters (all linear layers).

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), "CUDA GPU required (Kaggle: Runtime -> Change runtime type -> GPU T4 x2)"

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print("compute dtype:", compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    attn_implementation="sdpa",  # flash_attention_2 needs sm_80+; T4 is sm_75
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


In [ ]:
# 8) Train — QLoRA SFT with the corpus split as validation. (Kaggle T4: ~1-2 h.)

from trl import SFTTrainer

sft_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="epoch",
    fp16=not use_bf16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
)

try:  # modern trl: SFTConfig carries max_seq_length
    from trl import SFTConfig
    args = SFTConfig(**sft_kwargs, max_seq_length=MAX_SEQ_LEN, packing=False)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        formatting_func=format_for_sft,
    )
except ImportError:  # older trl: TrainingArguments + SFTTrainer(max_seq_length=...)
    from transformers import TrainingArguments
    args = TrainingArguments(**sft_kwargs)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        formatting_func=format_for_sft,
        max_seq_length=MAX_SEQ_LEN, packing=False,
    )

trainer.train()
TRAINED = True


In [ ]:
# 9) Post-training eval — strict-JSON accuracy on held-out eval_source.jsonl.
#    Scoring reuses the harness contract: extract_json() then per-key match.
#    (The final gate is scripts/eval_harness.py + Echidna on the VPS.)

def run_eval(model, tokenizer, records, n=EVAL_N_ROWS, max_new_tokens=MAX_NEW_TOKENS):
    sample = random.Random(SEED).sample(records, min(n, len(records)))
    results = []
    for r in sample:
        msgs = build_messages(r)
        gold = json.loads(msgs[-1]["content"])
        prompt = tokenizer.apply_chat_template(
            msgs[:-1], tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        out = model.generate(
            prompt, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        text = tokenizer.decode(out[0][prompt.shape[1]:], skip_special_tokens=True)
        try:
            row = score_answer(gold, extract_json(text))
        except Exception as e:
            row = {"json_ok": False, "vulnerable": False, "severity": False, "vuln_type": False,
                   "error": f"{type(e).__name__}: {str(e)[:120]}"}
        row["gold_severity"] = gold["severity"]
        if row["json_ok"]:
            row["pred_severity"] = extract_json(text).get("severity")
        results.append(row)
    return results

if "TRAINED" in globals() and TRAINED:
    results = run_eval(trainer.model, tokenizer, eval_records)
    n = len(results)
    print(f"evaluated {n} held-out records")
    for k in ("json_ok", "vulnerable", "severity", "vuln_type"):
        print(f"{k:12s}: {sum(r[k] for r in results)}/{n} = {100 * sum(r[k] for r in results) / n:.1f}%")
    bad = [r for r in results if not r["json_ok"]]
    print("non-JSON outputs:", len(bad), bad[0].get("error", "") if bad else "")
else:
    print("skip eval — model not trained in this session")


In [ ]:
# 10) Save the adapter (weights + tokenizer + config) — then merge/export on the VPS.

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("adapter saved to", OUTPUT_DIR)

# Optional GGUF export for Ollama/LM Studio (docs pipeline; needs Unsloth + more RAM):
#   !pip install -q unsloth
#   merged = model.merge_and_unload(); merged.save_pretrained(OUTPUT_DIR + "-merged")
#   # convert to GGUF, then: ollama create hunter -f <(vps_setup.sh Modelfile)


## After training

1. Download `/kaggle/working/hunter-qlora` (adapter + tokenizer).
2. On the VPS: merge the adapter into the base model, export GGUF (Unsloth), import into Ollama
   (`scripts/vps_setup.sh` scaffolds the whole stack: Ollama + Foundry + Echidna + Slither + Modelfile).
3. Gate candidates: `python3 scripts/eval_harness.py --model hunter --contract x.sol --function f`
   — `GATE=PASS` only when the patched contract compiles **and** Echidna observes a behavioral divergence.
4. Manual PoC review → Immunefi submission.

**Known corpus gaps (documented in ANALYSIS_REPORT.md):** no negative examples (`is_real` is always true),
`poc`/`fix` empty on ~80% of rows, `patched_function` is not trainable from the corpus (fixes are prose).
The eval cell therefore scores `vulnerable`/`severity`/`vuln_type`; the Echidna gate handles patch validation separately.
